# 07 — Constructors and Destructors

Constructors and destructors control the lifetime of objects — when they are born and when they die. A constructor runs automatically when an object is created; a destructor runs automatically when the object goes out of scope or is explicitly deleted. This automatic management is the foundation of C++'s approach to resource management (memory, file handles, network connections, etc.).


## The Default Constructor

A **default constructor** takes no arguments. If you define no constructors at all, the compiler generates one for you (it does nothing for built-in types, and calls the default constructor of member objects). If you define *any* constructor, the compiler no longer generates a default one.


In [ ]:
#include <iostream>
#include <string>

// No constructor defined — compiler generates a default one silently
class Widget {
public:
    int id;
    std::string label;
    // compiler-generated default constructor: does nothing for int,
    // calls std::string default constructor for label (sets to "")
};

// Explicit default constructor with initialization
class Gadget {
public:
    int id;
    std::string label;

    Gadget() {  // explicit default constructor
        id = 0;
        label = "unnamed";
        std::cout << "Gadget created (default)" << std::endl;
    }
};

Widget w;    // compiler default: id is uninitialized (garbage!), label is ""
w.id = 99;
std::cout << "Widget id: " << w.id << std::endl;

Gadget g;    // calls our default constructor
std::cout << "Gadget id: " << g.id << ", label: " << g.label << std::endl;
// Expected output:
// Widget id: 99
// Gadget created (default)
// Gadget id: 0, label: unnamed

## Parameterized Constructors

You can define constructors with parameters to initialize objects with specific values. Multiple constructors can coexist (function overloading).

The **constructor initialization list** (`: member(value), ...`) initializes members *before* the constructor body runs. This is the preferred form because:
- `const` and reference members **must** be initialized this way
- It's more efficient (no default-then-assign; just one construction)
- It makes the initialization order explicit


In [ ]:
#include <iostream>
#include <string>

class Box {
private:
    double width;
    double height;
    double depth;

public:
    // Using assignment in the body (less preferred)
    Box(double w, double h, double d) {
        width  = w;  // assignment (member is already default-constructed before this)
        height = h;
        depth  = d;
        std::cout << "Box created (assignment body): "
                  << w << "x" << h << "x" << d << std::endl;
    }

    double volume() const { return width * height * depth; }
};

class BoxV2 {
private:
    double width;
    double height;
    double depth;

public:
    // Using initialization list (preferred)
    BoxV2(double w, double h, double d)
        : width(w), height(h), depth(d)  // initialization list
    {
        // body only for logic that can't go in the list
        std::cout << "BoxV2 created (init list): "
                  << w << "x" << h << "x" << d << std::endl;
    }

    double volume() const { return width * height * depth; }
};

Box b1(3.0, 4.0, 5.0);
std::cout << "Volume: " << b1.volume() << std::endl;

BoxV2 b2(3.0, 4.0, 5.0);
std::cout << "Volume: " << b2.volume() << std::endl;
// Expected output:
// Box created (assignment body): 3x4x5
// Volume: 60
// BoxV2 created (init list): 3x4x5
// Volume: 60

**Exercise 1:** Create a class `Circle` with a private `radius` and a parameterized constructor. Add an `area()` method using `3.14159 * r * r`. Use an initialization list in the constructor.


In [ ]:
#include <iostream>

// Your code here

## Constructor Initialization List — Deep Dive

**Important rule:** Members are initialized in the **order they are declared in the class**, not the order they appear in the initialization list. A wrong order assumption can cause bugs.


In [ ]:
#include <iostream>

class Tricky {
private:
    int a;  // declared first, initialized first
    int b;  // declared second, initialized second

public:
    // Potential bug: list order says b first, but a is initialized first
    // because 'a' is declared before 'b' in the class
    Tricky(int val)
        : b(val), a(b * 2)  // a initialized first (uses b before b is set!)
    {}

    void print() const {
        std::cout << "a=" << a << ", b=" << b << std::endl;
    }
};

// What do you expect? Let's see:
Tricky t(5);
t.print();
// b is initialized to 5, but a = b*2 uses b BEFORE b is initialized
// (because a is declared first). This is undefined behavior!
// Expected output: b=5, but a=garbage or 0 (undefined)

std::cout << "--- Correct version ---" << std::endl;

In [ ]:
#include <iostream>

class Correct {
private:
    int a;  // initialized first
    int b;  // initialized second

public:
    Correct(int val)
        : a(val), b(a * 2)  // list order matches declaration order
    {}

    void print() const {
        std::cout << "a=" << a << ", b=" << b << std::endl;
    }
};

// const member: MUST use initialization list (cannot assign to const)
class ImmutableId {
private:
    const int id;     // must be set in init list
    std::string name;

public:
    ImmutableId(int i, std::string n) : id(i), name(n) {}

    void print() const {
        std::cout << "id=" << id << ", name=" << name << std::endl;
    }
};

Correct c(5);
c.print();  // a=5, b=10

ImmutableId obj(42, "Alice");
obj.print();
// Expected output:
// a=5, b=10
// id=42, name=Alice

## The Destructor

The destructor `~ClassName()` is called automatically when an object is destroyed (goes out of scope, or `delete` is called). It takes no arguments and has no return type. Its job is to release resources acquired in the constructor.


In [ ]:
#include <iostream>
#include <cstring>  // for strlen, strcpy

class ManagedBuffer {
private:
    char *data;
    int size;

public:
    ManagedBuffer(const char *str) {
        size = strlen(str) + 1;
        data = new char[size];   // allocate memory
        strcpy(data, str);
        std::cout << "ManagedBuffer created: \"" << data << "\"" << std::endl;
    }

    ~ManagedBuffer() {
        std::cout << "ManagedBuffer destroyed: \"" << data << "\"" << std::endl;
        delete[] data;  // release memory
        data = NULL;    // good practice: null the pointer after delete
    }

    void print() const {
        std::cout << "Buffer contains: \"" << data << "\"" << std::endl;
    }
};

{
    ManagedBuffer buf("Hello, C++!");
    buf.print();
    // buf goes out of scope here -> destructor is called automatically
}
std::cout << "After the block — buf is gone." << std::endl;
// Expected output:
// ManagedBuffer created: "Hello, C++!"
// Buffer contains: "Hello, C++!"
// ManagedBuffer destroyed: "Hello, C++!"
// After the block — buf is gone.

## Object Lifetime Visualization

Watch the order of construction and destruction as objects are created in nested scopes. Objects are destroyed in **reverse order** of construction (LIFO — like a stack).


In [ ]:
#include <iostream>
#include <string>

class Tracked {
private:
    std::string name;

public:
    Tracked(std::string n) : name(n) {
        std::cout << "  [+] " << name << " created" << std::endl;
    }

    ~Tracked() {
        std::cout << "  [-] " << name << " destroyed" << std::endl;
    }
};

std::cout << "--- Entering outer scope ---" << std::endl;
{
    Tracked outer("outer");

    std::cout << "  --- Entering inner scope ---" << std::endl;
    {
        Tracked inner1("inner1");
        Tracked inner2("inner2");
        std::cout << "  --- Leaving inner scope ---" << std::endl;
        // inner2 destroyed first, then inner1 (reverse order)
    }

    std::cout << "  Back in outer scope" << std::endl;
    // outer destroyed here
}
std::cout << "--- After outer scope ---" << std::endl;
// Expected output:
// --- Entering outer scope ---
//   [+] outer created
//   --- Entering inner scope ---
//   [+] inner1 created
//   [+] inner2 created
//   --- Leaving inner scope ---
//   [-] inner2 destroyed
//   [-] inner1 destroyed
//   Back in outer scope
//   [-] outer destroyed
// --- After outer scope ---

## Multiple Constructors

A class can have multiple constructors — this is just function overloading applied to constructors. The compiler picks the right one based on the argument types at the call site.


In [ ]:
#include <iostream>
#include <string>

class Player {
private:
    std::string name;
    int level;
    int health;

public:
    // Default constructor
    Player() : name("Unknown"), level(1), health(100) {
        std::cout << "Player default ctor" << std::endl;
    }

    // Name only
    Player(std::string n) : name(n), level(1), health(100) {
        std::cout << "Player name ctor: " << n << std::endl;
    }

    // Full specification
    Player(std::string n, int lvl, int hp) : name(n), level(lvl), health(hp) {
        std::cout << "Player full ctor: " << n << std::endl;
    }

    void print() const {
        std::cout << "  " << name << " | Level " << level << " | HP " << health << std::endl;
    }
};

Player p1;                          // default
Player p2("Alice");                 // name only
Player p3("Bob", 5, 250);          // full

p1.print();
p2.print();
p3.print();
// Expected output:
// Player default ctor
// Player name ctor: Alice
// Player full ctor: Bob
//   Unknown | Level 1 | HP 100
//   Alice | Level 1 | HP 100
//   Bob | Level 5 | HP 250

**Exercise 2:** Create a class `Vector2D` with `x` and `y` members. Write:
- A default constructor that sets both to 0
- A parameterized constructor that takes `x` and `y`
- A `print()` method that outputs `(x, y)`

Test both constructors.


In [ ]:
#include <iostream>

// Your code here

## Copy Constructor

The copy constructor creates a new object as a copy of an existing one. Syntax: `ClassName(const ClassName &other)`. It's called when:
- You pass an object by value to a function
- You return an object by value from a function
- You write `MyClass b = a;` (initialization, not assignment)

If your class manages raw memory (owns a pointer), the compiler-generated copy constructor does a **shallow copy** — both objects end up pointing to the same memory. This causes double-free bugs. The solution is a custom copy constructor (covered fully in the OCF notebook).


In [ ]:
#include <iostream>
#include <string>

class SimpleValue {
public:
    int value;

    SimpleValue(int v) : value(v) {
        std::cout << "Constructed with value=" << v << std::endl;
    }

    // Custom copy constructor
    SimpleValue(const SimpleValue &other) : value(other.value) {
        std::cout << "Copy constructed from value=" << other.value << std::endl;
    }
};

void printValue(SimpleValue v) {  // pass by value — triggers copy constructor
    std::cout << "Inside function, value=" << v.value << std::endl;
}

SimpleValue a(42);
SimpleValue b = a;   // copy constructor (initialization)
b.value = 99;

std::cout << "a.value=" << a.value << " (unchanged)" << std::endl;
std::cout << "b.value=" << b.value << std::endl;

printValue(a);  // copy constructor called again
// Expected output:
// Constructed with value=42
// Copy constructed from value=42
// a.value=42 (unchanged)
// b.value=99
// Copy constructed from value=42
// Inside function, value=42

## new and delete

`new` allocates memory on the **heap** and calls the constructor. `delete` calls the destructor and frees the memory. Objects created on the stack are destroyed automatically; heap objects must be deleted manually.


In [ ]:
#include <iostream>
#include <string>

class Resource {
private:
    std::string name;

public:
    Resource(std::string n) : name(n) {
        std::cout << "[+] Resource " << name << " acquired" << std::endl;
    }

    ~Resource() {
        std::cout << "[-] Resource " << name << " released" << std::endl;
    }

    void use() const {
        std::cout << "Using " << name << std::endl;
    }
};

// Stack allocation: destructor called automatically when block ends
std::cout << "--- Stack allocation ---" << std::endl;
{
    Resource stackObj("stack-resource");
    stackObj.use();
}  // destructor called here

// Heap allocation: we are responsible for calling delete
std::cout << "--- Heap allocation ---" << std::endl;
Resource *heapObj = new Resource("heap-resource");  // constructor called
heapObj->use();
delete heapObj;  // destructor called, memory freed
heapObj = NULL;  // good practice

std::cout << "--- Done ---" << std::endl;
// Expected output:
// --- Stack allocation ---
// [+] Resource stack-resource acquired
// Using stack-resource
// [-] Resource stack-resource released
// --- Heap allocation ---
// [+] Resource heap-resource acquired
// Using heap-resource
// [-] Resource heap-resource released
// --- Done ---

## Array of Objects

`new ClassName[n]` allocates an array of objects and calls the **default constructor** on each one. To delete an array, you **must** use `delete[]` (not `delete`). Using plain `delete` on an array is undefined behavior.


In [ ]:
#include <iostream>

class Point {
public:
    int x;
    int y;

    Point() : x(0), y(0) {}  // default constructor required for new[]

    Point(int px, int py) : x(px), y(py) {}
};

// Allocate an array of 5 Points (default constructor called 5 times)
Point *points = new Point[5];

// Initialize them manually
for (int i = 0; i < 5; i++) {
    points[i].x = i * 2;
    points[i].y = i * 3;
}

for (int i = 0; i < 5; i++) {
    std::cout << "points[" << i << "] = (" << points[i].x << ", " << points[i].y << ")" << std::endl;
}

delete[] points;  // MUST use delete[] for arrays, NOT delete
points = NULL;
// Expected output:
// points[0] = (0, 0)
// points[1] = (2, 3)
// points[2] = (4, 6)
// points[3] = (6, 9)
// points[4] = (8, 12)

**Exercise 3:** Allocate a `Circle` (from Exercise 1) on the heap. Call `area()` on it. Then `delete` it. Verify the destructor is called by printing a message inside the destructor.


In [ ]:
#include <iostream>

// Define Circle with a destructor that prints, then allocate on heap
// Your code here

## Final Exercise

Create a class `StringWrapper` that manages a dynamically allocated C-string (`char*`):

- **Constructor** `StringWrapper(const char *str)` — allocate a `new char[...]` and copy `str` into it
- **Destructor** — `delete[]` the buffer and print a message
- **`print()` method** — print the stored string
- **`length()` method** — return the string length (use `strlen`)

Test with:
1. A stack-allocated `StringWrapper` inside a block (verify destructor fires when block ends)
2. A heap-allocated `StringWrapper` (verify destructor fires when you `delete` it)


In [ ]:
#include <iostream>
#include <cstring>

// Your code here

## Modern C++ (C++11 and Beyond)


In [ ]:
#include <iostream>
#include <string>

// C++11: delegating constructors — one constructor calls another
class Config {
private:
    std::string host;
    int port;
    bool ssl;

public:
    // Primary constructor with all parameters
    Config(std::string h, int p, bool s)
        : host(h), port(p), ssl(s)
    {
        std::cout << "Config(" << host << ", " << port << ", " << ssl << ")" << std::endl;
    }

    // Delegating constructors: call the primary constructor
    Config() : Config("localhost", 80, false) {}          // no-arg defaults
    Config(std::string h) : Config(h, 80, false) {}       // host only
    Config(std::string h, int p) : Config(h, p, false) {} // host + port

    void print() const {
        std::cout << (ssl ? "https" : "http") << "://" << host << ":" << port << std::endl;
    }
};

Config c1;                           // defaults
Config c2("example.com");            // host only
Config c3("api.example.com", 443, true);  // all params

c1.print();
c2.print();
c3.print();

In [ ]:
#include <iostream>
#include <memory>  // for unique_ptr
#include <string>

class Widget {
public:
    std::string name;

    Widget() = default;             // C++11: explicitly ask for compiler default
    Widget(const Widget &) = delete; // C++11: forbid copying this class

    Widget(std::string n) : name(n) {
        std::cout << "Widget '" << name << "' created" << std::endl;
    }
    ~Widget() {
        std::cout << "Widget '" << name << "' destroyed" << std::endl;
    }
};

// C++11: unique_ptr — no manual delete needed
// Destructor is called automatically when unique_ptr goes out of scope
{
    std::unique_ptr<Widget> w(new Widget("smart"));
    std::cout << "Using widget: " << w->name << std::endl;
    // w->name etc work via ->
    // No need to call delete — unique_ptr does it automatically
}
std::cout << "After scope — widget was automatically cleaned up" << std::endl;
// Expected output:
// Widget 'smart' created
// Using widget: smart
// Widget 'smart' destroyed
// After scope — widget was automatically cleaned up